# 奖牌计数与首次获奖概率

参考 [2025 MCM C](https://www.contest.comap.com/undergraduate/contests/mcm/contests/2025/problems/2025_MCM_Problem_C.pdf)。
每行代表一个国家在一届比赛中的表现。示例使用固定随机种子的模拟数据，部分代表队在后期才参赛。留出最后一届。

泊松模型提供计数基线。负二项模型用离散度参数表达额外的变异。
两阶段门槛模型将有无获奖与获奖后的正计数分开建模，后者使用零截断分布。

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import brier_score_loss, mean_absolute_error
from statsmodels.discrete.truncated_model import HurdleCountModel

rng = np.random.default_rng(2025)
countries = 150
resources = rng.normal(size=countries)
first_edition = np.zeros(countries, dtype=int)
first_edition[-40:] = rng.integers(5, 12, size=40)
previously_medaled = np.zeros(countries, dtype=bool)
editions = []

for edition in range(12):
    entered = first_edition <= edition
    host = (np.arange(countries) == edition).astype(int)
    delegation = rng.poisson(np.exp(3.3 + 0.65 * resources))
    log_team = np.log1p(delegation) - 3.3
    win_probability = 1 / (1 + np.exp(-(-0.95 + 1.7 * log_team + 0.9 * host)))
    positive_mean = np.exp(1.2 + 0.7 * log_team + 0.6 * host)
    positive_medals = 1 + rng.negative_binomial(3, 3 / (3 + positive_mean))
    medals = np.where(rng.random(countries) < win_probability, positive_medals, 0)
    editions.append(pd.DataFrame({
        "edition": edition, "country": np.arange(countries), "host": host,
        "log_team": log_team, "prior_medal": previously_medaled.copy(), "medals": medals,
    }).loc[entered])
    previously_medaled |= entered & (medals > 0)

data = pd.concat(editions, ignore_index=True)
train = data[data["edition"] < 11]
test = data[data["edition"] == 11].copy()
features = ["log_team", "host"]
x_train = sm.add_constant(train[features], has_constant="add").to_numpy()
x_test = sm.add_constant(test[features], has_constant="add").to_numpy()
y_train = train["medals"].to_numpy()
y_test = test["medals"].to_numpy()

## 拟合三个模型

实际使用代表团规模时，先核对它是否在预测时点已知。
这里的首次获奖指从模拟历史起算，按每届开赛前的奖牌记录判定。
$P(Y>0)$ 用来评估首次获奖概率，计数误差用 MAE。

In [2]:
poisson = sm.GLM(y_train, x_train, family=sm.families.Poisson()).fit()
negbin = sm.NegativeBinomial(y_train, x_train).fit(disp=False)
hurdle = HurdleCountModel(y_train, x_train, dist="poisson", zerodist="poisson").fit(disp=False)

mean_poisson = poisson.predict(x_test)
mean_nb = negbin.predict(x_test)
alpha = negbin.params[-1]
forecasts = {
    "Poisson": (mean_poisson, -np.expm1(-mean_poisson)),
    "Negative binomial": (mean_nb, 1 - (1 + alpha * mean_nb) ** (-1 / alpha)),
    "Hurdle": (hurdle.predict(x_test), 1 - hurdle.predict(x_test, which="prob-zero")),
}

eligible = ~test["prior_medal"].to_numpy()
print(f"Eligible countries: {eligible.sum()} | first medals observed: {(y_test[eligible] > 0).sum()}")
pd.DataFrame([
    {
        "model": name,
        "medal MAE": mean_absolute_error(y_test, expected),
        "first-medal Brier": brier_score_loss(y_test[eligible] > 0, probability[eligible]),
        "expected first-medal countries": probability[eligible].sum(),
    }
    for name, (expected, probability) in forecasts.items()
]).round(3)

Eligible countries: 37 | first medals observed: 5


,model,medal MAE,first-medal Brier,expected first-medal countries
0,Poisson,2.250,0.125,14.790
1,Negative binomial,2.291,0.066,7.944
2,Hurdle,2.192,0.049,5.947


在真实题目里可逐届滚动回测，并检查强队与此前未获奖国家的误差。
各国预测独立求和时，若题目要求奖牌总数与项目设置一致，还需施加总量约束。